# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Environment Variables

In [28]:
import os

# Async CUDA allocator
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

# If cuDNN autotune fails, fall back to a safe (but slower) algorithm.
os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

# Allow TensorFlow to allocate GPU memory as needed
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true' 

### 1.2. Imports

In [29]:
from _imports import * # Centralized file containing all imports

### 1.3. GPU Management

In [30]:
# Specify GPU to use (e.g., GPU 0)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
troo.get_gpu_info()

TensorFlow Version: 2.18.0
CUDA support detected
  CUDA Version: 12.5.1
  cuDNN Version: 9

GPUs Detected (1): ['/physical_device:GPU:0']
Default GPU device: /device:GPU:0


I0000 00:00:1746447834.205485   20388 gpu_device.cc:2022] Created device /device:GPU:0 with 2229 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


## 2. Run Parameters 

In [ ]:
NUM_TRIALS = 1000
EPOCHS = 50
TOP_K = 5  # Number of top trials to save

mixed_precision.set_global_policy("mixed_float16")

#? Set to an existing path to resume training
RESUME_TRAINING_PATH = "runs/nas_2d_batool_1" # None or "runs/nas_1" 

In [32]:
RUN_DIR = RESUME_TRAINING_PATH or troo.create_run_directory(prefix="nas_")
print(f"Run directory: {RUN_DIR}")

Run directory: runs/nas_2d_batool_1


## 3. Data Loading and Preprocessing

In [33]:
def convert_to_sparse_labels(y: np.ndarray) -> np.ndarray:
    """
    Converts beam score targets to sparse integer labels suitable for SparseCategoricalCrossentropy loss.

    Args:
        y (np.ndarray): Original beam scores with shape (N, 8, 32).

    Returns:
        np.ndarray: Array of integer labels with shape (N,), where each label corresponds 
                    to the index (flattened over 8x32) of the maximum score.
    """
    # Reshape the input so that each sample becomes a 1D array (e.g., 256 elements)
    y_flat = y.reshape(y.shape[0], -1)
    # For each sample, return the index of the maximum value
    labels = np.argmax(y_flat, axis=1)
    return labels


In [34]:
# Define the base directory for data files
DATA_DIR = "./data/batool"

# ———————————————————————————— Load the train data ——————————————————————————— #
train_beam_output_path = os.path.join(DATA_DIR, "beam_output", "beams_output_train.npz")
train_coord_input_path = os.path.join(DATA_DIR, "coord_input", "coord_train.npz")
train_lidar_input_path = os.path.join(DATA_DIR, "lidar_input", "lidar_train.npz")

y_train = np.load(train_beam_output_path)

y_train = np.load(train_beam_output_path)['output_classification']
coord_input_train = np.load(train_coord_input_path)['coordinates']
lidar_input_train = np.load(train_lidar_input_path)['input']

# Cast target beam outputs to float - REMOVING USELESS IMAG PART
y_train = y_train.astype(np.float32)
coord_input_train = coord_input_train.astype(np.float32)
lidar_input_train = lidar_input_train.astype(np.float32)

print(f"Shape before conversion: {y_train.shape}")
y_train = convert_to_sparse_labels(y_train)
print(f"Shape after conversion: {y_train.shape}")

# Print the shapes of the loaded data
print(f"y_train shape: {y_train.shape}")
print(f"coord_input_train shape: {coord_input_train.shape}")
print(f"lidar_input_train shape: {lidar_input_train.shape}")

# ————————————————————————— Load the Validation Data ————————————————————————— #
val_beam_output_path = os.path.join(DATA_DIR, "beam_output", "beams_output_validation.npz")
val_coord_input_path = os.path.join(DATA_DIR, "coord_input", "coord_validation.npz")
val_lidar_input_path = os.path.join(DATA_DIR, "lidar_input", "lidar_validation.npz")

y_val = np.load(val_beam_output_path)['output_classification']
coord_input_val = np.load(val_coord_input_path)['coordinates']
lidar_input_val = np.load(val_lidar_input_path)['input']

y_val = y_val.astype(np.float32)
coord_input_val = coord_input_val.astype(np.float32)
lidar_input_val = lidar_input_val.astype(np.float32)

print(f"Shape before conversion: {y_val.shape}")
y_val = convert_to_sparse_labels(y_val)
print(f"Shape after conversion: {y_val.shape}")

print(f"y_val shape: {y_val.shape}")
print(f"coord_input_val shape: {coord_input_val.shape}")
print(f"lidar_input_val shape: {lidar_input_val.shape}")

# ———————————————————————————— Load the test Data ———————————————————————————— #
test_beam_output_path = os.path.join(DATA_DIR, "beam_output", "beams_output_test.npz")
test_coord_input_path = os.path.join(DATA_DIR, "coord_input", "coord_test.npz")
test_lidar_input_path = os.path.join(DATA_DIR, "lidar_input", "lidar_test.npz")

y_test = np.load(test_beam_output_path)['output_classification']
coord_input_test = np.load(test_coord_input_path)['coordinates']
lidar_input_test = np.load(test_lidar_input_path)['input']

y_test = y_test.astype(np.float32)
coord_input_test = coord_input_test.astype(np.float32)
lidar_input_test = lidar_input_test.astype(np.float32)

print(f"Shape before conversion: {y_test.shape}")
y_test = convert_to_sparse_labels(y_test)
print(f"Shape after conversion: {y_test.shape}")

print(f"y_test shape: {y_test.shape}")
print(f"coord_input_test shape: {coord_input_test.shape}")
print(f"lidar_input_test shape: {lidar_input_test.shape}")


# Final Summary of Data Shapes
print("\nFinal Summary of Data Shapes:")
print(f"Training Data: Lidar Input: {lidar_input_train.shape}, Coord Input: {coord_input_train.shape}, Labels: {y_train.shape}")
print(f"Validation Data: Lidar Input: {lidar_input_val.shape}, Coord Input: {coord_input_val.shape}, Labels: {y_val.shape}")
print(f"Test Data: Lidar Input: {lidar_input_test.shape}, Coord Input: {coord_input_test.shape}, Labels: {y_test.shape}")

/tmp/ipykernel_20388/3197121328.py:16: ComplexWarning: Casting complex values to real discards the imaginary part
  y_train = y_train.astype(np.float32)


Shape before conversion: (9234, 8, 32)
Shape after conversion: (9234,)
y_train shape: (9234,)
coord_input_train shape: (9234, 2)
lidar_input_train shape: (9234, 20, 200, 10)
Shape before conversion: (1960, 8, 32)
Shape after conversion: (1960,)
y_val shape: (1960,)
coord_input_val shape: (1960, 2)
lidar_input_val shape: (1960, 20, 200, 10)


/tmp/ipykernel_20388/3197121328.py:38: ComplexWarning: Casting complex values to real discards the imaginary part
  y_val = y_val.astype(np.float32)
/tmp/ipykernel_20388/3197121328.py:59: ComplexWarning: Casting complex values to real discards the imaginary part
  y_test = y_test.astype(np.float32)


Shape before conversion: (9638, 8, 32)
Shape after conversion: (9638,)
y_test shape: (9638,)
coord_input_test shape: (9638, 2)
lidar_input_test shape: (9638, 20, 200, 10)

Final Summary of Data Shapes:
Training Data: Lidar Input: (9234, 20, 200, 10), Coord Input: (9234, 2), Labels: (9234,)
Validation Data: Lidar Input: (1960, 20, 200, 10), Coord Input: (1960, 2), Labels: (1960,)
Test Data: Lidar Input: (9638, 20, 200, 10), Coord Input: (9638, 2), Labels: (9638,)


## 4. Getters

### 4.1. Regularizers

In [35]:
def get_regularizer(trial: optuna.Trial, name: str) -> Optional[tf.keras.regularizers.Regularizer]:
    """
    Suggests a regularization strategy using Optuna and returns the corresponding Keras regularizer.
    
    Args:
        trial (optuna.Trial): Optuna trial object used to sample the regularizer.
        name (str): Unique identifier for this regularizer parameter (used as key).

    Returns:
        Optional[tf.keras.regularizers.Regularizer]: The selected Keras regularizer instance,
        or `None` if "none" was selected.
    """
    # Suggest a regularizer type
    reg_type: str = trial.suggest_categorical(
        name,
        [
            "none",
            "l1",
            "l2",
            "l1l2",
            # "orthogonal",  #! only works for rank-2 tensors
        ],
    )

    # Map each regularizer name to a corresponding Keras regularizer instance
    regularizer_map: Dict[str, Optional[tf.keras.regularizers.Regularizer]] = {
        "none": None,
        "l1": regularizers.L1(l1=0.01),
        "l2": regularizers.L2(l2=0.01),
        "l1l2": regularizers.L1L2(l1=0.01, l2=0.01),
        "orthogonal": regularizers.OrthogonalRegularizer(factor=0.01, mode="rows"),
    }

    # Return the appropriate regularizer, or None if not found
    return regularizer_map.get(reg_type, None)

### 4.2. Activation Functions

In [36]:
def get_activation(trial: Any, name: str) -> Union[str, Callable[..., layers.Layer]]:
    """
    Suggests an activation function from a predefined list using Optuna.

    Args:
        trial (Any): The Optuna trial instance used to suggest a value.
        name (str): A unique name for this hyperparameter (e.g., "layer_1_activation").

    Returns:
        Union[str, Callable[..., layers.Layer]]: A string representing the activation function.
        This can be passed directly into a Keras layer's `activation=` argument.
    """
    return trial.suggest_categorical(
        name,
        [
            "relu",
            "tanh",
            "sigmoid",  # Logistic
            "elu", 
            "swish",  # x * sigmoid(x)
            "leaky_relu",
        ],
    )

### 4.3. Optimizers

In [37]:
def get_optimizer(trial: optuna.Trial) -> tf.keras.optimizers.Optimizer:
    """
    Suggests and returns a TensorFlow optimizer with a trial-based learning rate.

    Args:
        trial (optuna.Trial): Optuna trial object used for hyperparameter suggestion.

    Returns:
        tf.keras.optimizers.Optimizer: An instance of the selected optimizer.
    """
    # Suggest optimizer name from a predefined categorical set
    optimizer_name = trial.suggest_categorical(
        "optimizer",
        [
            "AdamW",
            "SGD",
            "Adam",
            "RMSprop",
            "Nadam",
            "Lion",
        ],
    )

    # Suggest learning rate on a logarithmic scale between 1e-5 and 1e-2
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)

    # Mapping of optimizer names to their TensorFlow classes
    optimizer_map: Dict[str, Type[tf.keras.optimizers.Optimizer]] = {
        "Adam": optimizers.Adam,
        "AdamW": optimizers.AdamW,
        "SGD": optimizers.SGD,
        "RMSprop": optimizers.RMSprop,
        "Nadam": optimizers.Nadam,
        "Lion": optimizers.Lion,
    }

    # Raise error if selected optimizer is not supported in the current context
    if optimizer_name not in optimizer_map:
        raise ValueError(
            f"Optimizer '{optimizer_name}' is not supported. "
            f"Supported optimizers are: {list(optimizer_map.keys())}."
        )

    # Instantiate and return the selected optimizer with suggested learning rate
    return optimizer_map[optimizer_name](learning_rate=learning_rate)

### 4.4. Callbacks

In [38]:
def get_callbacks(trial: optuna.Trial, checkpoint_dir: str) -> List[tf.keras.callbacks.Callback]:
    """
    Constructs and returns a list of Keras callbacks tailored for Optuna trials.

    Args:
        trial (optuna.Trial): The current Optuna trial object.
        checkpoint_dir (str): Directory where model weights will be saved.

    Returns:
        List[tf.keras.callbacks.Callback]: A list of callbacks to pass into `model.fit()`.
    """
    # Construct path for saving weights for this specific trial
    checkpoint_path: str = os.path.join(checkpoint_dir, f"trial_{trial.number}.weights.h5")

    # Metric to monitor for early stopping and checkpointing
    monitor: str = "val_loss"

    # Stop training early if no improvement in validation loss for N epochs
    early_stopping = callbacks.EarlyStopping(
        monitor=monitor,
        patience=6,  # number of epochs to wait
        restore_best_weights=True,
        verbose=1,
    )

    # Reduce learning rate if validation loss plateaus
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor=monitor,
        patience=3,  # how many epochs to wait before reducing LR
        factor=0.2,  # reduce LR by this factor
        min_lr=1e-6,  # don't reduce below this
        verbose=1,
    )

    # Save only the best model weights based on monitored metric
    model_checkpoint = callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor=monitor,
        save_best_only=True,  # only save weights if val_loss improves
        save_weights_only=True,  # save only the weights (not full model)
        verbose=0,
    )

    #! ——————— WARNING: the callbacks below do not work with multi-objective —————— !#
    # Custom callback to prune trial if NaN loss is encountered
    nan_pruner_callback = NanLossPrunerCallback(trial)

    # Optuna's built-in pruning callback for early trial termination
    pruning_callback = KerasPruningCallback(trial, monitor)
    #! ———————————————————————————————————————————————————————————————————————————— !#

    # Return the complete list of callbacks
    return [early_stopping, reduce_lr, model_checkpoint, nan_pruner_callback, pruning_callback]

### 4.5. Scalers

In [39]:
def get_scaler(
    trial: optuna.Trial,
) -> Union[StandardScaler, MinMaxScaler, RobustScaler, QuantileTransformer, PowerTransformer]:
    """
    Suggests and returns a scikit-learn scaler based on Optuna hyperparameter selection.

    Args:
        trial (optuna.Trial): Optuna trial object used to suggest hyperparameters.

    Returns:
        Union[StandardScaler, MinMaxScaler, RobustScaler, QuantileTransformer, PowerTransformer]:
            Instantiated scaler object from scikit-learn.
    """
    # Suggest a scaler name from the list of supported options
    scaler_name = trial.suggest_categorical(
        "scaler",
        [
            "StandardScaler",  # For normally-distributed data
            "MinMaxScaler_-1_1",  # Normalize to [-1, 1] range
            "MinMaxScaler_0_1",  # Normalize to [0, 1] range
            "RobustScaler",  # For data with outliers
            "QuantileTransformer",  # For non-normal or skewed data
            "PowerTransformer",  # For heavy-tailed or skewed data
        ],
    )

    # Return the appropriate scaler instance based on selection
    if scaler_name == "StandardScaler":
        return StandardScaler()
    elif scaler_name == "RobustScaler":
        return RobustScaler()
    elif scaler_name == "QuantileTransformer":
        return QuantileTransformer(output_distribution="normal")
    elif scaler_name == "PowerTransformer":
        return PowerTransformer(method="yeo-johnson")
    elif scaler_name == "MinMaxScaler_0_1":
        return MinMaxScaler(feature_range=(0, 1))
    elif scaler_name == "MinMaxScaler_-1_1":
        return MinMaxScaler(feature_range=(-1, 1))

    # Catch invalid or unknown choices
    else:
        raise ValueError(f"Unknown scaler selected: {scaler_name}")

## 5. Layers Builders

### 5.1. CNN

In [40]:
def build_cnn2d(
    trial: optuna.Trial,
    x: layers.Layer,
    num_layers: int = 5,
    max_filters: int = 256,
    min_filters: int = 32,
    filter_step: int = 32,
    max_kernel_size: int = 10,
    min_pool_size_dim1: int = 2,
    max_pool_size_dim1: int = 2,
    min_pool_size_dim2: int = 2,
    max_pool_size_dim2: int = 2,
    use_batch_norm: bool = False,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    custom_name: str = "cnn",
) -> layers.Layer:
    """
    Builds a 2D CNN where Optuna picks filters, kernels, and pooling window per layer.

    Args:
        trial: Optuna trial object.
        x: Input Keras tensor.
        num_layers: Number of Conv2D+Pool blocks.
        max_filters: Upper bound on filters.
        min_filters: Lower bound on filters.
        filter_step: Step size for filters.
        max_kernel_size: Max kernel dim for height & width.
        min_pool_size_dim1: Min pooling window height.
        max_pool_size_dim1: Max pooling window height.
        min_pool_size_dim2: Min pooling window width.
        max_pool_size_dim2: Max pooling window width.
        use_batch_norm: If True, trial.prunes BatchNorm on/off.
        use_regularization: If True, trial selects kernel/bias/activity regularizers.
        residual_method: One of {None, "beside", "all"}.
        custom_name: Prefix for naming each layer.

    Returns:
        The output tensor after all blocks.
    """

    # Containers for residual strategies
    beside_residual: Optional[layers.Layer] = None
    all_skip_connections: List[layers.Layer] = []

    for layer_idx in range(num_layers):
        # 0) Sample a single pooling window to use in every block
        pool_dim1 = trial.suggest_int(
            f"{custom_name}_pool_size_dim1_{layer_idx}",
            min_pool_size_dim1,
            max_pool_size_dim1,
        )
        pool_dim2 = trial.suggest_int(
            f"{custom_name}_pool_size_dim2_{layer_idx}",
            min_pool_size_dim2,
            max_pool_size_dim2,
        )
        pool_size: Tuple[int, int] = (pool_dim1, pool_dim2)
        
        # 1) Filters
        num_filters = trial.suggest_int(
            f"{custom_name}_filters_layer_{layer_idx}",
            min_filters,
            max_filters,
            step=filter_step,
        )

        # 2) Kernel dims
        kernel_h = trial.suggest_int(f"{custom_name}_kernel_height_{layer_idx}", 1, max_kernel_size)
        kernel_w = trial.suggest_int(f"{custom_name}_kernel_width_{layer_idx}", 1, max_kernel_size)

        # Activation
        activation_fn = get_activation(trial, f"{custom_name}_activation_layer_{layer_idx}")

        # Regularizers
        kernel_reg = (
            get_regularizer(trial, f"{custom_name}_kernel_regularizer_layer_{layer_idx}")
            if use_regularization
            else None
        )
        bias_reg = (
            get_regularizer(trial, f"{custom_name}_bias_regularizer_layer_{layer_idx}")
            if use_regularization
            else None
        )
        activity_reg = (
            get_regularizer(trial, f"{custom_name}_activity_regularizer_layer_{layer_idx}")
            if use_regularization
            else None
        )

        # Conv2D
        x = layers.Conv2D(
            filters=num_filters,
            kernel_size=(kernel_h, kernel_w),
            activation=activation_fn,
            padding="same",
            name=f"{custom_name}_conv2d_{layer_idx}",
            kernel_regularizer=kernel_reg,
            bias_regularizer=bias_reg,
            activity_regularizer=activity_reg,
        )(x)

        # 3) Optional BatchNorm
        if use_batch_norm and trial.suggest_categorical(
            f"{custom_name}_use_batch_norm_layer_{layer_idx}", [True, False]
        ):
            x = layers.BatchNormalization(name=f"{custom_name}_batch_norm_{layer_idx}")(x)

        # 4) Residuals
        if residual_method == "beside":
            if layer_idx == 0:
                beside_residual = x
            else:
                if trial.suggest_categorical(f"{custom_name}_use_residual_layer_{layer_idx}", [True, False]):
                    prev = beside_residual
                    target_ch = x.shape[-1]
                    if prev.shape[-1] != target_ch:
                        prev = layers.Conv2D(
                            filters=target_ch,
                            kernel_size=(1, 1),
                            padding="same",
                            name=f"{custom_name}_res_align_{layer_idx}",
                        )(prev)
                    x = layers.Add(name=f"{custom_name}_res_add_{layer_idx}")([x, prev])
                    beside_residual = x
                else:
                    beside_residual = x

        elif residual_method == "all":
            if layer_idx == 0:
                all_skip_connections = [x]
            else:
                to_add: List[layers.Layer] = []
                for prev_idx, prev_layer in enumerate(all_skip_connections):
                    if trial.suggest_categorical(
                        f"{custom_name}_use_residual_layer_{layer_idx}_{prev_idx}",
                        [True, False],
                    ):
                        prev = prev_layer
                        target_ch = x.shape[-1]
                        if prev.shape[-1] != target_ch:
                            prev = layers.Conv2D(
                                filters=target_ch,
                                kernel_size=(1, 1),
                                padding="same",
                                name=(f"{custom_name}_skip_res_conv2d_" f"{layer_idx}_{prev_idx}"),
                            )(prev)
                        to_add.append(prev)
                if to_add:
                    x = layers.Add(name=f"{custom_name}_res_all_add_{layer_idx}")([x] + to_add)
                all_skip_connections.append(x)

        # 5) MaxPooling with sampled window
        x = layers.MaxPooling2D(pool_size=pool_size, name=f"{custom_name}_maxpool_{layer_idx}")(x)

    return x

## 6. Objective Function

In [41]:
def objective(
    trial: optuna.Trial,
    X: List[np.ndarray],
    y: List[np.ndarray],
    checkpoint_dir: str,
    model_dir: str,
    fig_dir: str,
    logs_dir: str,
    epochs: int = 50,
    size_penalizer: Optional[str] = None,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    show_summary: bool = False,
    plot_model: bool = False,
) -> float:
    """
    Objective function for Optuna to optimize a Neural Network NN on any-input data.

    Args:
        trial (optuna.Trial): Current trial for hyperparameter suggestions.
        X (List[np.ndarray]): List of input arrays.
        y (List[np.ndarray]): List of label arrays.
        checkpoint_dir (str): Path to store checkpoint files.
        model_dir (str): Path to store full models.
        fig_dir (str): Path to store plots.
        logs_dir (str): Path to store logs.
        epochs (int): Number of training epochs.
        size_penalizer (Optional[str]): type of penalizer to use:
            - "params": Penalizes based on the number of parameters.
            - "flops": Penalizes based on the number of FLOPs.
            - None: No penalization is applied.
        use_regularization (bool): If True, adds regularization (e.g., L1/L2) to layers to prevent overfitting.
        residual_method (Optional[str]): tyoe of residual connection to use:
            - "beside": Adds residual connections between consecutive layers.
            - "all": test residual connections between all layers.
            - None: No residual connections are applied.
        show_summary (bool): If True, display the model summary.
        plot_model (bool): If True, display a plot of the model architecture.

    Returns:
        float: Final validation loss (optionally penalized) used for optimization.
    """

    # Each trial gets a different seed to split the data
    np.random.seed(trial.number)
    tf.random.set_seed(trial.number)

    # ————————————————————————————— Prepare the Data ————————————————————————————— #
    x_lidar_train = X[0]
    x_coord_train = X[1]
    x_lidar_val = X[2]
    x_coord_val = X[3]
    
    y_train = y[0]
    y_val = y[1]

    # ———————————————————————————————————————————————————————————————————————————— #

    model = None
    try:

        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Model Construction                              #
        # ———————————————————————————————————————————————————————————————————————————— #

        # —————————————————————————————————— Scaler —————————————————————————————————— #
        # scaler = get_scaler(trial)
        # x_coord_train = scaler.fit_transform(x_coord_train)
        # x_coord_val = scaler.transform(x_coord_val)

        # ——————————————————————————————— LiDAR Input ——————————————————————————————— #
        # Input for LiDAR data (e.g., shape: (20, 200, 10))
        x_lidar_input = layers.Input(shape=(20, 200, 10))

        # ———————————————————————————————— GPS Input ———————————————————————————————— #
        # Input for coordinate data (e.g., shape: (2,))
        x_coord_input = layers.Input(shape=(x_coord_train.shape[1],))

        # Use z-score normalization
        norm_layer = layers.Normalization(axis=1, name="coord_input_normalization")

        # Computes the mean and variance of the input data
        norm_layer.adapt(x_coord_train)

        # Apply normalization to the input data
        x_coord_norm = norm_layer(x_coord_input)

        # Add spatial dimension
        x_coord_norm = layers.Reshape((1, 1, x_coord_input.shape[1]))(x_coord_norm)

        # Tile across the lidar grid
        # So the coordinates are repeated across the 20x200 grid
        x_coord_norm = layers.Lambda(lambda x: tf.tile(x, [1, 20, 200, 1]))(x_coord_norm)

        # ? If using scaler then uncomment the following line
        # x_coord_input = layers.Reshape((1, 1, x_coord_input.shape[1]))(x_coord_input)
        # ————————————————————————————— Combine Branches ————————————————————————————— #
        # Fuse channels: (batch,20,200,10) + (batch,20,200,2) → (batch,20,200,12)
        combined = layers.Concatenate(axis=-1)([x_lidar_input, x_coord_norm])

        max_layers = trial.suggest_int("cnn_num_layers", 1, 4)

        # Calculate max pool size
        max_pool_dim1 = math.floor(20 ** (1.0 / max_layers))
        max_pool_dim2 = math.floor(200 ** (1.0 / max_layers))

        x = build_cnn2d(
            trial=trial,
            x=combined,
            num_layers=max_layers,
            max_filters=128,
            min_filters=16,
            filter_step=4,
            max_kernel_size=7,
            min_pool_size_dim1=1,
            max_pool_size_dim1=max_pool_dim1,
            min_pool_size_dim2=1,
            max_pool_size_dim2=max_pool_dim2,
            use_batch_norm=True,
            use_regularization=use_regularization,
            residual_method=residual_method,
        )
        # ———————————————————————————— Flatten the Output ———————————————————————————— #
        x = layers.Flatten(name="flatten")(x)

        # ——————————————————————————————— Dense Layers ——————————————————————————————— #
        num_dense_layers = trial.suggest_int("num_dense_layers", 0, 2)
        for i in range(num_dense_layers):
            # Suggest the number of units for each dense layer
            units = trial.suggest_int(f"dense_{i+1}_units", 64, 512, step=64)
            x = layers.Dense(
                units=units,
                activation=get_activation(trial, f"dense_{i+1}_activation"),
                name=f"dense_{i+1}",
            )(x)
            x = layers.Dropout(rate=0.5)(x)

        # —————————————————————————————————— Output —————————————————————————————————— #
        outputs = layers.Dense(256, activation="softmax")(x)

        # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
        model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

        # ———————————————————————————— Vizualize the Model ——————————————————————————— #
        if show_summary:
            model.summary()

        if plot_model:
            # Display the model architecture image
            tf.keras.utils.plot_model(
                model,
                to_file=os.path.join(fig_dir, f"model_plot_{trial.number}.png"),
                show_shapes=True,
                show_layer_names=True,
            )
            display(Image(filename=os.path.join(fig_dir, f"model_plot_{trial.number}.png")))

        # ————————————————————————————— Compile the Model ———————————————————————————— #
        optimizer = get_optimizer(trial)
        model.compile(
            optimizer=optimizer,
            loss=losses.SparseCategoricalCrossentropy(),
            metrics=["accuracy"],
        )

        # ———————————————————————————————— Train Model ——————————————————————————————— #
        batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])
        history = model.fit(
            [x_lidar_train, x_coord_train],
            y_train,
            validation_data=([x_lidar_val, x_coord_val], y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=get_callbacks(trial, checkpoint_dir),
            verbose=2,
        )

        model.save(os.path.join(model_dir, f"trial_{trial.number}.keras"))

        # ———————————————————————————————————————————————————————————————————————————— #
        #                            Penalize the Model Size                           #
        # ———————————————————————————————————————————————————————————————————————————— #
        loss = min(history.history["val_loss"])
        if size_penalizer == "flops":
            loss = troo.compute_flops_penalized_loss(loss=loss, model=model)
        elif size_penalizer == "params":
            loss = troo.compute_params_penalized_loss(loss=loss, model=model)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                                 Trial Results                                #
        # ———————————————————————————————————————————————————————————————————————————— #
        clear_output(wait=True)

        epochs = list(range(1, len(history.history["loss"]) + 1))
        train_loss = history.history["loss"]
        val_loss = history.history["val_loss"]
        train_acc = history.history.get("accuracy", [])
        val_acc = history.history.get("val_accuracy", [])

        # Create figure with two subplots
        fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(16, 6))

        # Left: Loss
        ax_loss.plot(epochs, train_loss, marker="o", linestyle="-", label="Training Loss")
        ax_loss.plot(epochs, val_loss, marker="x", linestyle="--", label="Validation Loss")
        ax_loss.set_title("Training & Validation Loss")
        ax_loss.set_xlabel("Epoch")
        ax_loss.set_ylabel("Loss")
        ax_loss.set_xticks(epochs)
        ax_loss.set_ylim(0, max(max(train_loss), max(val_loss)) * 1.05)
        ax_loss.grid(True)
        ax_loss.legend()

        # Right: Accuracy (if available)
        if train_acc and val_acc:
            ax_acc.plot(epochs, train_acc, marker="v", linestyle="-", label="Training Accuracy")
            ax_acc.plot(epochs, val_acc, marker="^", linestyle="--", label="Validation Accuracy")
            ax_acc.set_title("Training & Validation Accuracy")
            ax_acc.set_xlabel("Epoch")
            ax_acc.set_ylabel("Accuracy")
            ax_acc.set_xticks(epochs)
            ax_acc.set_ylim(0, 1)
            ax_acc.grid(True)
            ax_acc.legend()

            trial.set_user_attr("best_train_accuracy", float(max(train_acc)))
            trial.set_user_attr("best_val_accuracy", float(max(val_acc)))
        else:
            ax_acc.axis("off")  # hide if accuracy not present

        fig.tight_layout()
        fig.savefig(os.path.join(fig_dir, f"trial_{trial.number}.png"), dpi=300)
        plt.close(fig)

        # ————————————————————————————— Evaluate on s009 ————————————————————————————— #
        test_loss, test_acc = model.evaluate(
            [lidar_input_test, coord_input_test], y_test, batch_size=batch_size, verbose=0
        )

        trial.set_user_attr("test_accuracy_s009", float(test_acc))

        # ————————————————————————————— Print the results ———————————————————————————— #

        print(f"\n\n# ——————————————————————— Trial {trial.number} Results ——————————————————————— #")
        print("\n" + "=" * 15)
        print(f"Training loss: {loss:.12f}")
        print(f"Training accuracy: {max(train_acc):.4f}\n")
        print(f"Validation loss: {loss:.12f}")
        print(f"Validation accuracy: {max(val_acc):.4f}\n")
        print(f"Test loss (s009):     {test_loss:.12f}")
        print(f"Test accuracy (s009): {test_acc:.4f}\n")
        
        params = model.count_params()
        print(f"Number of parameters: {params}")
        print(f"Model size: {params * 4 / (1024 ** 2):.2f} MB")
        print("=" * 15 + "\n")
        print("# ———————————————————————————————————————————————————————————————————————————— #\n\n")

        return loss

    except optuna.exceptions.TrialPruned:
        raise  # simply propagate pruning
    except tf.errors.ResourceExhaustedError as oom_err:
        # Catch OOM / resource exhausted
        print(f"❌ Trial {trial.number} hit OOM (ResourceExhaustedError): {oom_err}")

        # Log the error to a file in the logs directory
        error_log_path = os.path.join(logs_dir, f"trial_{trial.number}_error.log")
        with open(error_log_path, "w") as log_file:
            log_file.write(f"Trial {trial.number} encountered an error:\n")
            log_file.write(str(oom_err) + "\n\n")
            log_file.write("Traceback:\n")
            traceback.print_exc(file=log_file)

        return float("inf")  # Return bad loss
    except Exception as e:
        print(f"An error occurred during the trial execution: {e}")
        traceback.print_exc()

        # Log the error to a file in the logs directory
        error_log_path = os.path.join(logs_dir, f"trial_{trial.number}_error.log")
        with open(error_log_path, "w") as log_file:
            log_file.write(f"Trial {trial.number} encountered an error:\n")
            log_file.write(str(e) + "\n\n")
            log_file.write("Traceback:\n")
            traceback.print_exc(file=log_file)

        return float("inf")  # Return bad loss
    finally:
        if model is not None:
            clear_session()
            del model

## 7. Code Health Check

In [42]:
# resources_dir = os.path.join(RUN_DIR, "resources")
# os.makedirs(resources_dir, exist_ok=True)
# troo.log_resources(log_dir=resources_dir)

In [43]:
try:
    pid = os.getpid()
    cmd = (
        f'python3 "{os.path.abspath("_monitor_kernel_life.py")}" '
        f"--pid {pid} --custom-title {RUN_DIR}; exec bash"
    )
    terminals = [
        ["xfce4-terminal", "--disable-server", "--hold", "-e", f'bash -c "{cmd}"'],
        ["gnome-terminal", "--disable-factory", "--", "bash", "-i", "-c", cmd],
        ["xterm", "-hold", "-e", cmd],
        ["konsole", "--hold", "-e", f'bash -c "{cmd}"'],
    ]
    term = next((t for t in terminals if shutil.which(t[0])), None)
    if not term:
        raise RuntimeError(
            "No supported terminal emulator found; install gnome-terminal, "
            "xfce4-terminal, konsole, or xterm."
        )
    _monitor_proc = subprocess.Popen(term, preexec_fn=os.setpgrp)
    print(f"[INFO] Launched monitor in {term[0]} (PID={pid})")
except Exception as e:
    print(f"[ERROR] Auto launching kernel monitoring failed! {e}\n")
    display(
        HTML(
            f"Call the monitor script manually: "
            f'<span style="color: orange;">'
            f"python _monitor_kernel_life.py --pid {pid} --custom-title {RUN_DIR}"
            f"</span>"
        )
    )
    pass

[INFO] Launched monitor in gnome-terminal (PID=20388)


## Main

In [44]:
try:
    # ——————————————————————————————— Storage paths —————————————————————————————— #
    study_dir = os.path.join(RUN_DIR, "optuna_study")
    os.makedirs(study_dir, exist_ok=True)

    dirs = {
        "args": os.path.join(study_dir, "args"),
        "figures": os.path.join(study_dir, "figures"),
        "weights": os.path.join(study_dir, "weights"),
        "models": os.path.join(study_dir, "models"),
        "logs": os.path.join(study_dir, "logs"),
    }
    for path in dirs.values():
        os.makedirs(path, exist_ok=True)

    storage_path = f"sqlite:///{os.path.join(study_dir, 'optuna_study.db')}"
    checkpoint_dir, model_dir, fig_dir, args_dir, logs_dir = (
        dirs["weights"],
        dirs["models"],
        dirs["figures"],
        dirs["args"],
        dirs["logs"],
    )

    print(f"Initializing study at '{study_dir}'...")

    # —————————————————————————————————— Pruners ————————————————————————————————— #
    pruner = optuna.pruners.HyperbandPruner()

    # ——————————————————————————————————— Study —————————————————————————————————— #
    study = optuna.create_study(
        study_name=os.path.basename(study_dir),
        storage=storage_path,
        direction="minimize",
        pruner=pruner,
        load_if_exists=True,
    )

    # Count trials done, then determine the remaining trials
    done_trials = len(
        study.get_trials(
            deepcopy=False,
            states=(
                optuna.trial.TrialState.COMPLETE,
                optuna.trial.TrialState.PRUNED,
                optuna.trial.TrialState.FAIL,
            ),
        )
    )
    n_remaining_trials = max(0, NUM_TRIALS - done_trials)

    study.optimize(
        lambda trial: objective(
            trial,
            X=[lidar_input_train, coord_input_train, lidar_input_val, coord_input_val],
            y=[y_train, y_val],
            checkpoint_dir=checkpoint_dir,
            model_dir=model_dir,
            fig_dir=fig_dir,
            logs_dir=logs_dir,
            epochs=EPOCHS,
            size_penalizer=None,
            use_regularization=False,
            residual_method=None,  #! Find your backbone first
            show_summary=False,
        ),
        n_trials=n_remaining_trials,
        catch=(ValueError, RuntimeError),
        gc_after_trial=True,
        n_jobs=1,  # If you have multiple GPUs/Cores
        show_progress_bar=False,
    )

    # ————————————————————————————— Save Top-K Trials ———————————————————————————— #
    valid_trials = [
        t for t in study.trials
        if t.value is not None and not (math.isnan(t.value) or math.isinf(t.value))
    ]
    sorted_trials = sorted(valid_trials, key=lambda t: t.value)[:TOP_K]

    for rank, trial in enumerate(sorted_trials):
        trial_id = trial.number
        trial_params = trial.params
        trial_loss = trial.value
        trial_train_acc = trial.user_attrs.get("best_train_accuracy", None)
        trial_val_acc = trial.user_attrs.get("best_val_accuracy", None)
        trial_test_acc = trial.user_attrs.get("test_accuracy_s009", None)

        troo.save_trial_params_to_file(
            filepath=os.path.join(args_dir, f"top_{rank + 1}_trial.txt"),
            params=trial_params,
            rank=rank + 1,
            trial_id=trial_id,
            loss=trial_loss,
            val_accuracy=trial_val_acc,
            train_accuracy=trial_train_acc,
            test_accuracy=trial_test_acc,
            sampler=study.sampler.__class__.__name__,
        )

    # —————————————————————————— Clean-Up Non-Top Trials ————————————————————————— #
    all_trial_ids = {t.number for t in study.trials}
    top_trial_ids = {t.number for t in sorted_trials}

    cleanup_paths = [
        (checkpoint_dir, "trial_{trial_id}.weights.h5"),
        (model_dir, "trial_{trial_id}.keras"),
        (fig_dir, "trial_{trial_id}.png"),
    ]

    for trial_id in all_trial_ids - top_trial_ids:
        for base_dir, filename_template in cleanup_paths:
            file_path = os.path.join(base_dir, filename_template.format(trial_id=trial_id))
            if os.path.exists(file_path):
                os.remove(file_path)

    troo.analyze_study(study, fig_dir=fig_dir, table_dir=study_dir)

    # ————————————————————————————— End The Training ————————————————————————————— #
    failed_trials = sum(1 for t in study.trials if t.state != optuna.trial.TrialState.COMPLETE)
    notify_training_success(
        recipients_file="./json/recipients.json",
        credentials_file="./json/credentials.json",
        subject=f"🎉 Training Complete - Failed Trials: {failed_trials}",
    )
except Exception as e:
    print(f"An error occurred: {e}")
    traceback.print_exc()

[I 2025-05-05 09:24:45,948] Trial 1 finished with value: 4.801176071166992 and parameters: {'cnn_num_layers': 3, 'cnn_pool_size_dim1_0': 1, 'cnn_pool_size_dim2_0': 5, 'cnn_filters_layer_0': 24, 'cnn_kernel_height_0': 1, 'cnn_kernel_width_0': 4, 'cnn_activation_layer_0': 'sigmoid', 'cnn_use_batch_norm_layer_0': True, 'cnn_pool_size_dim1_1': 2, 'cnn_pool_size_dim2_1': 4, 'cnn_filters_layer_1': 20, 'cnn_kernel_height_1': 4, 'cnn_kernel_width_1': 5, 'cnn_activation_layer_1': 'relu', 'cnn_use_batch_norm_layer_1': False, 'cnn_pool_size_dim1_2': 1, 'cnn_pool_size_dim2_2': 2, 'cnn_filters_layer_2': 72, 'cnn_kernel_height_2': 4, 'cnn_kernel_width_2': 5, 'cnn_activation_layer_2': 'elu', 'cnn_use_batch_norm_layer_2': False, 'num_dense_layers': 1, 'dense_1_units': 128, 'dense_1_activation': 'leaky_relu', 'optimizer': 'Nadam', 'learning_rate': 0.0025446127413552644, 'batch_size': 64}. Best is trial 0 with value: 4.777022361755371.




# ——————————————————————— Trial 1 Results ——————————————————————— #

Training loss: 4.801176071167
Training accuracy: 0.2375

Validation loss: 4.801176071167
Validation accuracy: 0.1949

Test loss (s009):     4.669242382050
Test accuracy (s009): 0.1469

Number of parameters: 533721
Model size: 2.04 MB

# ———————————————————————————————————————————————————————————————————————————— #


Number of failed trials: 0



/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


,Parameter,Mean,Std,Min,25%,Median,75%,Max
0,params_batch_size,64.000000,0.000000,64.000000,64.000000,64.000000,64.000000,64.000000
1,params_cnn_filters_layer_0,66.000000,59.396970,24.000000,45.000000,66.000000,87.000000,108.000000
2,params_cnn_filters_layer_1,48.000000,39.597980,20.000000,34.000000,48.000000,62.000000,76.000000
3,params_cnn_filters_layer_2,50.000000,31.112698,28.000000,39.000000,50.000000,61.000000,72.000000
4,params_cnn_kernel_height_0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
5,params_cnn_kernel_height_1,3.500000,0.707107,3.000000,3.250000,3.500000,3.750000,4.000000
6,params_cnn_kernel_height_2,4.500000,0.707107,4.000000,4.250000,4.500000,4.750000,5.000000
7,params_cnn_kernel_width_0,3.000000,1.414214,2.000000,2.500000,3.000000,3.500000,4.000000
8,params_cnn_kernel_width_1,5.500000,0.707107,5.000000,5.250000,5.500000,5.750000,6.000000
9,params_cnn_kernel_width_2,3.000000,2.828427,1.000000,2.000000,3.000000,4.000000,5.000000


,Parameter,Category,Fraction,Count
0,params_cnn_activation_layer_0,swish,0.5,1
1,params_cnn_activation_layer_0,sigmoid,0.5,1
2,params_cnn_activation_layer_1,relu,1.0,2
3,params_cnn_activation_layer_2,elu,1.0,2
4,params_dense_1_activation,tanh,0.5,1
5,params_dense_1_activation,leaky_relu,0.5,1
6,params_dense_2_activation,swish,1.0,1
7,params_optimizer,Nadam,1.0,2


...

,Parameter,Category,Fraction,Count
0,params_cnn_activation_layer_0,swish,1.0,1
1,params_cnn_activation_layer_1,relu,1.0,1
2,params_cnn_activation_layer_2,elu,1.0,1
3,params_dense_1_activation,tanh,1.0,1
4,params_dense_2_activation,swish,1.0,1
5,params_optimizer,Nadam,1.0,1


/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


,Parameter,Mean,Std,Min,25%,Median,75%,Max
0,params_batch_size,64.000000,NaN,64.000000,64.000000,64.000000,64.000000,64.000000
1,params_cnn_filters_layer_0,24.000000,NaN,24.000000,24.000000,24.000000,24.000000,24.000000
2,params_cnn_filters_layer_1,20.000000,NaN,20.000000,20.000000,20.000000,20.000000,20.000000
3,params_cnn_filters_layer_2,72.000000,NaN,72.000000,72.000000,72.000000,72.000000,72.000000
4,params_cnn_kernel_height_0,1.000000,NaN,1.000000,1.000000,1.000000,1.000000,1.000000
5,params_cnn_kernel_height_1,4.000000,NaN,4.000000,4.000000,4.000000,4.000000,4.000000
6,params_cnn_kernel_height_2,4.000000,NaN,4.000000,4.000000,4.000000,4.000000,4.000000
7,params_cnn_kernel_width_0,4.000000,NaN,4.000000,4.000000,4.000000,4.000000,4.000000
8,params_cnn_kernel_width_1,5.000000,NaN,5.000000,5.000000,5.000000,5.000000,5.000000
9,params_cnn_kernel_width_2,5.000000,NaN,5.000000,5.000000,5.000000,5.000000,5.000000


,Parameter,Category,Fraction,Count
0,params_cnn_activation_layer_0,sigmoid,1.0,1
1,params_cnn_activation_layer_1,relu,1.0,1
2,params_cnn_activation_layer_2,elu,1.0,1
3,params_dense_1_activation,leaky_relu,1.0,1
4,params_optimizer,Nadam,1.0,1


[INFO] Email sent successfully.


In [45]:
# Kill the monitor kernel life process
if _monitor_proc is not None and _monitor_proc.poll() is None:
    os.killpg(_monitor_proc.pid, signal.SIGINT)